# COST — Paired Statistical Significance of Learned vs Random Gating
**No training.** This notebook loads existing per-patient predictions from the completed targeted notebook: `gating_ON_seed{42,43,44}_predictions.npz` and `r2_trained_random_mask_seed{42,43,44}.npz`.

Tests: (1) paired DeLong and paired stratified patient bootstrap for **each seed**; (2) bootstrap of the **mean of the three seed-wise AUROC differences** with the same patient resampling across seeds. The latter assesses held-out-patient uncertainty **conditional on these three trained seeds**, not generalization across all random initializations. A significant patient-level test would not by itself prove a practically meaningful gain. Do not confuse AUROC(mean predictions) with mean(seed AUROCs).


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, average_precision_score
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
BASE = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/revision2_outputs')
RANDOM = BASE / 'reviewer2_targeted_completion'
SEEDS = [42, 43, 44]
B = 2000
RNG_SEED = 20261107
# Expected original test cohort: one retained ICU stay per patient, n=1044.

scores = {}
for s in SEEDS:
    learned_path = BASE / f'gating_ON_seed{s}_predictions.npz'
    random_path  = RANDOM / f'r2_trained_random_mask_seed{s}.npz'
    assert learned_path.is_file(), f'Missing learned-gating predictions: {learned_path}'
    assert random_path.is_file(), f'Missing trained-random predictions: {random_path}'
    with np.load(learned_path, allow_pickle=False) as a, np.load(random_path, allow_pickle=False) as b:
        y1, y2 = a['test_labels'].astype(int), b['test_labels'].astype(int)
        assert np.array_equal(y1,y2), f'Seed {s}: labels/order differ — STOP'
        assert len(y1)==1044, f'Seed {s}: expected 1044 test rows, got {len(y1)}'
        assert set(np.unique(y1))=={0,1}
        learned, control = a['probs_ensemble'].astype(float), b['probs_ensemble'].astype(float)
        assert learned.shape==control.shape==y1.shape
        assert np.all(np.isfinite(learned)) and np.all(np.isfinite(control))
        assert np.all((learned>=0)&(learned<=1)) and np.all((control>=0)&(control<=1))
        scores[s] = (y1,learned,control)
y=scores[42][0]
for s in SEEDS[1:]:
    assert np.array_equal(y,scores[s][0]), f'Seed {s}: row alignment inconsistent — STOP'
print('PASS: same test labels and row order in all six prediction files; n =',len(y),'deaths =',int(y.sum()))
print('NOTE: NPZ predictions do not themselves contain patient IDs; this check relies on each file preserving the exact same original test cohort order.')


In [ ]:
# Paired DeLong implementation (same approach as original reviewer notebook)
def midrank(x):
    order=np.argsort(x,kind='mergesort'); z=np.asarray(x)[order]; n=len(z)
    v=np.empty(n,float); i=0
    while i<n:
        j=i+1
        while j<n and z[j]==z[i]: j+=1
        v[i:j]=(i+j-1)/2+1
        i=j
    out=np.empty(n,float);out[order]=v
    return out

def paired_delong(y,a,b):
    order=np.argsort(-y,kind='mergesort'); m=int(y.sum()); n=len(y)-m
    pred=np.vstack((a,b))[:,order]
    tx=np.vstack([midrank(z[:m]) for z in pred]);ty=np.vstack([midrank(z[m:]) for z in pred]);tz=np.vstack([midrank(z) for z in pred])
    aucs=tz[:,:m].sum(axis=1)/(m*n)-(m+1)/(2*n)
    v01=(tz[:,:m]-tx)/n;v10=1-(tz[:,m:]-ty)/m
    cov=np.cov(v01)/m+np.cov(v10)/n
    variance=float(np.array([1.,-1.])@cov@np.array([1.,-1.]))
    assert variance>=0, variance
    z=float((aucs[0]-aucs[1])/np.sqrt(variance)) if variance>0 else np.nan
    p=float(2*norm.sf(abs(z))) if variance>0 else np.nan
    return float(aucs[0]-aucs[1]),p

rows=[]
for s in SEEDS:
    ys,a,b=scores[s]
    da,p=paired_delong(ys,a,b)
    rows.append({'seed':s,'learned_AUROC':roc_auc_score(ys,a),'random_AUROC':roc_auc_score(ys,b),
                 'delta_AUROC':da,'paired_DeLong_p':p,
                 'learned_AUPRC':average_precision_score(ys,a),
                 'random_AUPRC':average_precision_score(ys,b)})
seed_results=pd.DataFrame(rows)
display(seed_results.round(6))
print('Mean per-seed AUROC difference =',seed_results.delta_AUROC.mean())


In [ ]:
# Stratified paired bootstrap: one matched test-patient sample per replicate,
# shared across all three seeds. This is NOT a bootstrap over independent training runs.
rng=np.random.default_rng(RNG_SEED)
pos=np.where(y==1)[0];neg=np.where(y==0)[0]
base_deltas=seed_results.delta_AUROC.to_numpy()
boot=np.empty((B,len(SEEDS)),float)
for k in range(B):
    idx=np.r_[rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)]
    yy=y[idx]
    for j,s in enumerate(SEEDS):
        _,a,b=scores[s]
        boot[k,j]=roc_auc_score(yy,a[idx])-roc_auc_score(yy,b[idx])

def boot_summary(x):
    lo,hi=np.percentile(x,[2.5,97.5]);p=min(1.,2*min(np.mean(x<=0),np.mean(x>=0)))
    return float(lo),float(hi),float(p)
for j,s in enumerate(SEEDS):
    lo,hi,p=boot_summary(boot[:,j]);seed_results.loc[j,'bootstrap_CI_lower']=lo
    seed_results.loc[j,'bootstrap_CI_upper']=hi;seed_results.loc[j,'bootstrap_p_two_sided']=p
mean_boot=boot.mean(axis=1)
mean_delta=float(base_deltas.mean());lo,hi,p=boot_summary(mean_boot)
print('Three-seed MEAN paired AUROC difference (learned − trained random):',f'{mean_delta:+.6f}')
print('95% paired stratified patient-bootstrap CI (conditional on 3 trained seeds):',f'[{lo:+.6f}, {hi:+.6f}]')
print('Two-sided bootstrap sign p (conditional on 3 trained seeds):',f'{p:.4f}')
print('Per-seed paired tests:')
display(seed_results.round(6))
RANDOM.mkdir(parents=True,exist_ok=True)
seed_results.to_csv(RANDOM/'R2_learned_vs_trained_random_per_seed_significance.csv',index=False)
pd.DataFrame([{'mean_delta':mean_delta,'CI_2.5':lo,'CI_97.5':hi,'bootstrap_p':p,'bootstrap_replicates':B,'seeds':'42,43,44','method':'paired stratified patient bootstrap; seed-wise AUROC deltas averaged'}]).to_csv(RANDOM/'R2_learned_vs_trained_random_mean_significance.csv',index=False)
print('Saved two statistical audit CSV files to',RANDOM)
print('IMPORTANT: Interpret CI and p conditional on these three fitted seed models; do not claim superiority across arbitrary random initializations.')


## Reporting and manuscript decision

- Report the **actual** Δ, CI and p from the executed notebook, not an invented or predicted significance level.
- If CI includes 0, report no statistically established advantage. If CI excludes 0, report a patient-level paired advantage *conditional on three trained seeds* while acknowledging small effect size and large 4.06× latency overhead.
- Keep existing gating ON vs OFF p=0.52 unchanged: it tests a **different comparison**.
- The mean of individual-seed AUROC values is the manuscript's `0.7996` vs `0.7923`; do not substitute AUROC of averaged probabilities without labeling it differently.
